# Market Mistral 7B QLoRA Training

This notebook fine-tunes `mistralai/Mistral-7B-Instruct-v0.3` for the Market/Volatility Agent using the prepared market JSONL dataset.

The dataset contains Yahoo Finance features and Gold risk labels. The output is a LoRA adapter saved to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U transformers datasets peft accelerate bitsandbytes sentencepiece

In [ ]:
from pathlib import Path
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/298A')
DATA_PATH = DRIVE_ROOT / 'data/gold/market_sentiment_finetune_data_clean.jsonl'
TRAINER_PATH = DRIVE_ROOT / 'agents/market/train_mistral_7b.py'
OUTPUT_DIR = DRIVE_ROOT / 'models/market_mistral_7b_lora'

assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH}'
assert TRAINER_PATH.exists(), f'Missing trainer: {TRAINER_PATH}'

rows = [json.loads(line) for line in DATA_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
labels = []
for row in rows:
    assistant = row['messages'][-1]['content']
    try:
        labels.append(json.loads(assistant).get('market_risk_label', 'UNKNOWN'))
    except json.JSONDecodeError:
        labels.append('INVALID_JSON')

from collections import Counter
print('Dataset:', DATA_PATH)
print('Examples:', len(rows))
print('Labels:', Counter(labels))
print('First roles:', [message['role'] for message in rows[0]['messages']])

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU not detected. Select Runtime > Change runtime type > T4 GPU.')

print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

## Train

This uses 4-bit QLoRA, a batch size of 1, gradient accumulation of 8, a 15% validation split, and three epochs.

In [ ]:
import subprocess

command = [
    'python3', str(TRAINER_PATH),
    '--train-jsonl', str(DATA_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--base-model', 'mistralai/Mistral-7B-Instruct-v0.3',
    '--epochs', '3',
    '--max-length', '1024',
    '--batch-size', '1',
    '--grad-accum', '8',
    '--learning-rate', '0.0001',
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
print('Saved files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'{path.name}: {path.stat().st_size / 1024 / 1024:.2f} MB')

metrics_path = OUTPUT_DIR / 'training_metrics.json'
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
else:
    raise FileNotFoundError(metrics_path)

## Output location

The adapter is saved at:

`/content/drive/MyDrive/298A/models/market_mistral_7b_lora`

Copy this adapter to the local project `models/` directory or use it from a Colab endpoint.